## Creating simple agent with Tracing

In [2]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [3]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

In [ ]:
# https://openai.github.io/openai-agents-python/agents/

carbon_accounting_agent = Agent(
    name = "Carbon Accounting Expert Assistant",
    instructions = """
    You are a helpful assistant giving out carbon accounting advice.
    You give concise answer.
    """
)

### Let's execute the Agent:

In [7]:
# https://openai.github.io/openai-agents-python/tracing/
# https://platform.openai.com/logs?api=traces
with trace("Carbon Accounting Expert Assistant"):
    result = await Runner.run(carbon_accounting_agent,"How to calculate emission on bitcoin mining infrastructure?")
print(result)
print(type(result))

RunResult:
- Last agent: Agent(name="Carbon Accounting Expert Assistant", ...)
- Final output (str):
    Here’s a concise way to estimate CO2e from a Bitcoin mining setup.
    
    Key inputs
    - IT power draw (kW): sum of all mining hardware watts (rated or measured).
    - PUE (power usage effectiveness): ratio of total facility power to IT power (dimensionless).
    - Annual hours: 8760 (or actual uptime).
    - Grid emission factor (EF): kg CO2e per kWh for the electricity you use (region/time-specific).
    
    Formula (annual emissions)
    - IT energy = IT_power_kW × 8760
    - Total facility energy = IT_energy × PUE
    - Emissions (kg CO2e/year) = Total facility energy × EF
      or equivalently: Emissions = (IT_power_kW × 8760 × PUE × EF)
    
    Notes
    - If you have only facility electricity use (not IT), use Emissions = Total_facility_kWh × EF.
    - EF should reflect the actual electricity mix your grid uses (e.g., regional grid factor or hourly factors if available

### Streaming the answer to the screen, token by token

In [8]:
response_stream = Runner.run_streamed(carbon_accounting_agent, "what is emission factor? explain for kids in a simple way using analogy")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Emission factor is like a recipe that tells you how much pollution comes from doing something a little bit. It’s the amount of emissions per unit of activity.

Analogy: If you bake cookies, the recipe says how many cookies you get from a cup of flour. Similarly, an emission factor says how much pollution you get per each unit of activity—like per kilometer driven, per liter of fuel used, or per kilowatt-hour of electricity.

Example: If a car emits 120 grams of CO2 per kilometer, the emission factor is 120 g CO2 per km.